## Local Chemical Environment Consistency

In [1]:
from types import SimpleNamespace
import sys
sys.path.append('../')

from pymatgen.core import Structure

from auxi_func import (
    charge_balance_from_structure,
    structure_add_atoms,
    structure_to_tensor,
)

from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import (
    mkdir, 
    get_pymatgen_structure,
    get_pbc_distances,
)

mkdir('../data/host/')
device = 'cuda:0'

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Folder exists


### 01 Structure Information

In [2]:
# Load supercell
s_super = Structure.from_file(
    filename = "../data/supercell.cif",
    site_tolerance = 0,
    frac_tolerance = 0,
)

# Load subcell
s_sub = Structure.from_file(
    filename = "../data/cut_subcell.cif",
    site_tolerance = 0,
    frac_tolerance = 0,
)

In [3]:
print(f"Composition of supercell: {s_super.composition}")
print(f"Composition of subcell: {s_sub.composition}")

print(f"Oxi state of supercell: {s_super.composition.oxi_state_guesses()}")
print(f"Oxi state of subcell: {s_sub.composition.oxi_state_guesses()}")

Composition of supercell: Li30 Ta30 Cl180
Composition of subcell: Li10 Ta7 Cl56


Oxi state of supercell: ({'Li': 1.0, 'Ta': 5.0, 'Cl': -1.0},)
Oxi state of subcell: []


### 02 Chemical Consistency (Charge Balance)

In [4]:
# Compute the oxidation state
oxi_state_super = s_super.composition.oxi_state_guesses()[0]
print(f"Oxidation state {oxi_state_super}")

Oxidation state {'Li': 1.0, 'Ta': 5.0, 'Cl': -1.0}


In [5]:
# Compute the charge balance
atoms_to_add = charge_balance_from_structure(
    structure = s_sub,
    oxi_state = oxi_state_super,
    verbose = True,
)

print(f"Number of atoms to be added: {atoms_to_add}")

11 positive charge need to be added to the subcell to match charge balance
{'Li': 1.0, 'Ta': 2.0, 'Cl': -0.0}
Number of atoms to be added: {'Li': 1.0, 'Ta': 2.0, 'Cl': -0.0}


### 03 Manipulate Subcell

In [6]:
s_temp = s_sub.copy()
print(len(s_temp))

# Add atoms to the subcell
s_temp = structure_add_atoms(
    structure = s_temp,
    atoms_to_add = atoms_to_add,
    verbose=True,
)
print(len(s_temp))

73
Atom Li added at position [5.93848839 2.56855487 0.61760924] (dis: 6.38)
Atom Ta added at position [2.23502699 1.17643247 4.4276626 ] (dis: 6.32)
Atom Ta added at position [11.38949081  9.21998483  2.54609283] (dis: 7.17)
76


### 04 Convert Subcell

In [7]:
(   
    lengths, 
    angles, 
    frac_coords, 
    atom_types, 
    num_atoms, 
    atom_masks
) = structure_to_tensor(
    structure = s_temp,
    cut_threshold = 5.0,
)

# Map to device
lengths, angles, frac_coords, atom_types, num_atoms, atom_masks = map(
    lambda x: x.to(device),
    (lengths, angles, frac_coords, atom_types, num_atoms, atom_masks),
)

print(f"Lengths: {lengths}")
print(f"Angles: {angles}")
print(f"Fractional coordinates: {frac_coords}")
print(f"Atom types: {atom_types}")
print(f"Number of atoms: {num_atoms}")
print(f"Atom masks: {atom_masks}")

Lengths: tensor([[12., 12., 12.]], device='cuda:0')
Angles: tensor([[90., 90., 90.]], device='cuda:0')
Fractional coordinates: tensor([[0.5000, 0.5000, 0.5000],
        [0.1689, 0.3042, 0.2053],
        [0.4577, 0.2620, 0.2839],
        [0.5488, 0.0999, 0.9593],
        [0.3970, 0.6255, 0.7200],
        [0.3836, 0.8119, 0.4279],
        [0.1542, 0.6834, 0.5059],
        [0.3364, 0.5154, 0.0936],
        [0.1307, 0.8862, 0.3537],
        [0.7286, 0.5869, 0.0193],
        [0.6955, 0.1710, 0.0805],
        [0.9030, 0.5841, 0.3837],
        [0.5940, 0.7536, 0.1944],
        [0.2808, 0.3584, 0.7637],
        [0.7073, 0.8710, 0.9015],
        [0.0686, 0.0547, 0.3773],
        [0.8226, 0.3719, 0.7766],
        [0.5592, 0.3457, 0.4236],
        [0.9595, 0.0286, 0.6812],
        [0.2316, 0.7169, 0.3214],
        [0.8172, 0.9308, 0.5484],
        [0.9946, 0.9157, 0.1122],
        [0.9328, 0.3050, 0.6466],
        [0.5641, 0.5739, 0.6632],
        [0.7404, 0.7703, 0.7537],
        [0.5519, 0.8604

### 05 Inpainting with Generative Model

In [8]:
# Load pre-trained model.
chggen = CHGGen.load_from_checkpoint('../../gen_org/data/test_models/mp/cutoff-7_epoch=49-val_loss=0.83.ckpt', strict=False, map_location=device)

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/utilities/migration/utils.py:55: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.1.4, which is newer than your current Lightning version: v2.0.9
  rank_zero_warn(
/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [9]:
# Print how many atoms are masked and no masked
print(f"Number of masked atoms to diffusion: {sum(atom_masks)}/{len(atom_masks)}")

Number of masked atoms to diffusion: 54/76


In [10]:
# Sampling.
ld_kwargs = SimpleNamespace(
    n_step_each = 5,
    num_noise_level = 200,
    signal_to_noise_ratio = 0.4,
    save_traj = False,
    disable_bar = False,                     
)

results = chggen.conditional_langevin_dynamics(
    lengths = lengths,
    angles = angles,
    composition = atom_types,
    num_atoms = num_atoms,
    ori_frac_coords = frac_coords,
    mask = atom_masks,
    ld_kwargs = ld_kwargs,
)

# repeats = len(results['all_frac_coords'])//results['num_atoms'][0]
repeats = 1
lengths = results['lengths'].repeat(repeats,1)
angles = results['angles'].repeat(repeats,1)
num_atoms = results['num_atoms'].repeat(repeats)
frac_coords = results['frac_coords']
atom_types = results['atom_types']

s_list = get_pymatgen_structure(
    lengths = lengths,         
    angles = angles,
    num_atoms = num_atoms,
    frac_coords = frac_coords,
    atom_types = atom_types,
)

  0%|          | 0/959 [00:00<?, ?it/s]

100%|██████████| 959/959 [00:50<00:00, 18.99it/s]


In [11]:
for i, structure in enumerate(s_list):
    structure.to(filename='../data/host/structure_' + str(i) + '.cif')